# Review Streaming (Kafka) - Functionized

Each function is defined in its own cell for easy conversion into a PySpark job later.


## Imports and shared Spark types


In [5]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.storagelevel import StorageLevel


## get_env: read environment variables with defaults


In [6]:
def get_env(name: str, default: str) -> str:
    """Return the environment variable value or a default."""
    return os.getenv(name, default)


## build_spark: create a SparkSession


In [7]:
def build_spark(mongo_uri: str = None) -> SparkSession:
    """Create a SparkSession with Mongo settings."""
    app_name = get_env('SPARK_APP_NAME', 'review-enrichment')
    shuffle_partitions = int(get_env('SPARK_SHUFFLE_PARTITIONS', '16'))
    builder = SparkSession.builder.appName(app_name)
    builder = builder.config("spark.sql.shuffle.partitions", str(shuffle_partitions))
    if mongo_uri:
        builder = (
            builder
            .config("spark.mongodb.read.connection.uri", mongo_uri)
            .config("spark.mongodb.write.connection.uri", mongo_uri)
        )
    return builder.getOrCreate()


## review_schema: schema for review JSON payloads


In [8]:
def review_schema() -> StructType:
    """Return a StructType describing the review JSON payload."""
    return StructType([
        StructField('review_id', StringType(), True),
        StructField('user_id', StringType(), True),
        StructField('business_id', StringType(), True),
        StructField('stars', IntegerType(), True),
        StructField('useful', IntegerType(), True),
        StructField('funny', IntegerType(), True),
        StructField('cool', IntegerType(), True),
        StructField('text', StringType(), True),
        StructField('date', StringType(), True),
    ])


## build_reviews_stream: read Kafka and parse JSON


In [9]:
def build_reviews_stream(spark: SparkSession):
    """Read Kafka and parse JSON payloads into a streaming DataFrame."""
    kafka_bootstrap = get_env('KAFKA_BOOTSTRAP_SERVERS', 'broker:29092')
    kafka_topic = get_env('KAFKA_TOPIC_REVIEW', 'raw_data_review')

    # Optional throttling for teaching/debugging (commented by default).
    # max_offsets = int(get_env('KAFKA_MAX_OFFSETS_PER_TRIGGER', '1'))
    # trigger_seconds = int(get_env('KAFKA_TRIGGER_SECONDS', '2'))

    kafka_df = (
        spark.readStream.format('kafka')
        .option('kafka.bootstrap.servers', kafka_bootstrap)
        .option('subscribe', kafka_topic)
        .option('startingOffsets', 'latest')
        # .option('maxOffsetsPerTrigger', max_offsets)
        .load()
    )

    schema = review_schema()
    return (
        kafka_df.selectExpr('CAST(value AS STRING) AS json_str')
        .select(from_json(col('json_str'), schema).alias('review'))
        .select('review.*')
    )


## Mongo helpers: read users and businesses


In [10]:
def build_mongo_uri() -> str:
    """Build a MongoDB connection URI from env vars."""
    host = get_env('MONGO_HOST', 'mongodb')
    port = get_env('MONGO_PORT', '27017')
    user = get_env('MONGO_USER', get_env('MONGO_INITDB_ROOT_USERNAME', 'root'))
    password = get_env('MONGO_PASSWORD', get_env('MONGO_INITDB_ROOT_PASSWORD', 'password'))
    auth_db = get_env('MONGO_AUTH_DB', 'admin')
    return f'mongodb://{user}:{password}@{host}:{port}/{auth_db}?authSource={auth_db}'

def load_mongo_collection(spark: SparkSession, collection: str, database: str = None):
    """Load a MongoDB collection into a DataFrame."""
    db = database or get_env('MONGO_DB', 'yelp')
    uri = build_mongo_uri()
    return (
        spark.read.format('mongodb')
        .option('database', db)
        .option('collection', collection)
        .option('uri', uri)
        .load()
    )

def load_users(spark: SparkSession):
    """Load users from MongoDB."""
    return load_mongo_collection(spark, 'users')

def load_businesses(spark: SparkSession):
    """Load businesses from MongoDB."""
    return load_mongo_collection(spark, 'businesses')


## show_streamed: view rows from the memory table


In [11]:
def show_streamed(spark: SparkSession, limit: int = 20):
    """Display the most recent rows captured by the memory sink."""
    df = spark.table('reviews_stream')
    df.orderBy(df['date'].desc()).show(limit, truncate=False)


## show_streamed_count: view counts of rows from the memory table

In [12]:
def show_streamed_count(spark: SparkSession):
    """Display the number of rows captured in the memory sink."""
    count = spark.table("reviews_stream").count()
    print(f"reviews_stream count: {count}")


## load_enriched_reviews: read enriched output from MongoDB


In [13]:
def load_enriched_reviews(spark: SparkSession):
    """Load enriched reviews from MongoDB."""
    return load_mongo_collection(
        spark,
        collection=get_env('MONGO_COLLECTION_REVIEWS_ENRICHED', 'reviews_enriched')
    )


## stop_all_streams: stop active streaming queries


In [14]:
def stop_all_streams(spark: SparkSession):
    """Stop every active streaming query attached to this Spark session."""
    for stream in spark.streams.active:
        stream.stop()


## reset_checkpoint: remove streaming checkpoint directory


In [15]:
def reset_checkpoint(path: str) -> None:
    """Delete a streaming checkpoint directory (stop streams first)."""
    if not path:
        return
    if os.path.exists(path):
        # shutil.rmtree can remove non-empty directories.
        import shutil
        shutil.rmtree(path)


## load_users/load_businesses: manual helpers for inspection


In [16]:
def load_users_df(spark: SparkSession):
    """Load users DataFrame for manual inspection."""
    return load_users(spark)

def load_businesses_df(spark: SparkSession):
    """Load businesses DataFrame for manual inspection."""
    return load_businesses(spark)

def show_user_samples(users_df, limit: int = 5):
    """Display a small user sample for sanity checks."""
    users_df.select('user_id', 'name', 'review_count', 'average_stars').show(limit, truncate=False)

def show_business_samples(businesses_df, limit: int = 5):
    """Display a small business sample for sanity checks."""
    businesses_df.select('business_id', 'name', 'city', 'stars').show(limit, truncate=False)


## prep_dims: narrow + dedupe user/business dimensions


In [17]:
def prep_dims(users_df, businesses_df):
    """Select only needed columns and drop duplicate IDs."""
    users_clean = (
        users_df
        .select(col("user_id"), col("name").alias("user_name"))
        # .dropDuplicates(["user_id"])
    )
    businesses_clean = (
        businesses_df
        .select(col("business_id"), col("name").alias("business_name"))
        # .dropDuplicates(["business_id"])
    )
    # Optional: repartition dims for join balance. This does NOT filter keys.
    # users_clean = users_clean.repartition("user_id")
    # businesses_clean = businesses_clean.repartition("business_id")
    # Optional: persist when you have enough memory/disk; otherwise skip.
    # Note: persist() is lazy, so count() forces it to load before streaming starts.
    users_clean = users_clean.persist(StorageLevel.DISK_ONLY)
    businesses_clean = businesses_clean.persist(StorageLevel.DISK_ONLY)
    users_clean.count()
    businesses_clean.count()
    return users_clean, businesses_clean


## build_enriched_reviews_df: join reviews with user/business


In [18]:
def build_enriched_reviews_df(reviews_stream, users_clean, businesses_clean):
    """Join streaming reviews with user/business dims and shape enriched schema."""
    reviews_pruned = reviews_stream.select(
        "review_id", "user_id", "business_id",
        "stars", "useful", "funny", "cool", "text", "date"
    )

    # Optional: repartition the stream for better balance on large batches.
    reviews_pruned = reviews_pruned.repartition("user_id")

    joined = (
        reviews_pruned
        .join(users_clean, on="user_id", how="left")
        .join(businesses_clean, on="business_id", how="left")
    )

    return joined.select(
        col("review_id"),
        col("user_id"),
        col("business_id"),
        col("date").alias("timestamp"),
        struct("stars", "useful", "funny", "cool", "text", "date").alias("review"),
        struct(col("user_name").alias("name")).alias("user_info"),
        struct(col("business_name").alias("name")).alias("business_info"),
    )


## write_enriched_to_mongo: foreachBatch upsert into reviews_enriched


In [19]:
def write_enriched_to_mongo(batch_df, batch_id: int):
    """Append enriched reviews to MongoDB (append-only)."""
    if batch_df.rdd.isEmpty():
        return

    # Optional: repartition to increase parallel writes when you have more workers.
    # Example: if you have 4 workers, try repartition(4) and tune from there.
    # batch_df = batch_df.repartition(4, 'review_id')

    db_name = get_env('MONGO_DB', 'yelp')
    collection_name = get_env('MONGO_COLLECTION_REVIEWS_ENRICHED', 'reviews_enriched')

    (batch_df
        .write
        .format('mongodb')
        .option('database', db_name)
        .option('collection', collection_name)
        .mode('append')
        .save()
    )


## start_enriched_stream: run foreachBatch writer


In [20]:
def start_enriched_stream(reviews_stream, users_clean, businesses_clean):
    """Start streaming write to MongoDB (reviews_enriched) using foreachBatch."""
    enriched = build_enriched_reviews_df(reviews_stream, users_clean, businesses_clean)
    # trigger_seconds = int(get_env('KAFKA_TRIGGER_SECONDS', '2'))
    writer = (
        enriched.writeStream
        .foreachBatch(write_enriched_to_mongo)
        .outputMode("append")
        .option('checkpointLocation', '/tmp/reviews_enriched_checkpoint')
    )
    # writer = writer.trigger(processingTime=f"{trigger_seconds} seconds")
    return writer.start()


## write_raw_reviews_to_mongo: foreachBatch append to reviews


In [21]:
def write_raw_reviews_to_mongo(batch_df, batch_id: int):
    """Append raw review records to MongoDB (reviews collection)."""
    if batch_df.rdd.isEmpty():
        return

    db = get_env('MONGO_DB', 'yelp')
    collection = get_env('MONGO_COLLECTION_REVIEW', 'reviews')
    (
        batch_df.write.format('mongodb')
        .mode('append')
        .option('database', db)
        .option('collection', collection)
        .save()
    )


## start_raw_review_stream: run foreachBatch writer (no enrichment)


In [22]:
def start_raw_review_stream(reviews_stream):
    """Start streaming write to MongoDB (reviews) without enrichment."""
    checkpoint = get_env('KAFKA_REVIEWS_RAW_CHECKPOINT', '/tmp/reviews_raw_checkpoint')
    # trigger_seconds = int(get_env('KAFKA_TRIGGER_SECONDS', '2'))
    writer = (
        reviews_stream.writeStream
        .foreachBatch(write_raw_reviews_to_mongo)
        .outputMode('append')
        .option('checkpointLocation', checkpoint)
    )
    # writer = writer.trigger(processingTime=f"{trigger_seconds} seconds")
    return writer.start()


## load_raw_reviews: read raw reviews from MongoDB


In [23]:
def load_raw_reviews(spark: SparkSession):
    """Load raw review documents from MongoDB."""
    return load_mongo_collection(
        spark,
        collection=get_env('MONGO_COLLECTION_REVIEW', 'reviews')
    )


## Optional: start_memory_sink (commented for non-testing runs)


In [24]:
# def start_memory_sink(reviews_stream, query_name: str = 'reviews_stream'):
#     """Write the stream to an in-memory table for ad-hoc inspection."""
#     trigger_seconds = getattr(reviews_stream, '_trigger_seconds', None)
#     writer = (
#         reviews_stream.writeStream
#         .format('memory')
#         .queryName(query_name)
#         .outputMode('append')
#     )
#     if trigger_seconds:
#         writer = writer.trigger(processingTime=f'{trigger_seconds} seconds')
#     return writer.start()


## Manual run: reset checkpoints (run after stopping streams)


In [25]:
reset_checkpoint('/tmp/reviews_raw_checkpoint')
reset_checkpoint('/tmp/reviews_enriched_checkpoint')


## Manual run: build Spark session


In [26]:
mongo_uri = build_mongo_uri()
spark = build_spark(mongo_uri=mongo_uri)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/05 16:36:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Manual run: load users + businesses (Mongo)


In [27]:
users_df = load_users_df(spark)
businesses_df = load_businesses_df(spark)
users_clean, businesses_clean = prep_dims(users_df, businesses_df)


## Manual run: build Kafka review stream


In [28]:
reviews_stream = build_reviews_stream(spark)


## Manual run: start raw review stream to MongoDB (non-enriched)


Run either the raw stream or the enriched stream below (not both at once).


In [29]:
# Uncomment to write raw reviews directly to MongoDB
# raw_query = start_raw_review_stream(reviews_stream)
# raw_query


## Manual run: optional memory sink (testing only)


In [30]:
# query = start_memory_sink(reviews_stream)


## Manual run: start enriched stream to MongoDB


In [31]:
enriched_query = start_enriched_stream(reviews_stream, users_clean, businesses_clean)
enriched_query


26/01/05 16:39:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Show enriched reviews from MongoDB (run after manual steps)


In [ ]:
enriched_df = load_enriched_reviews(spark)
enriched_df.show(5, truncate=False)


## Show raw reviews from MongoDB (run after raw stream starts)


In [ ]:
raw_df = load_raw_reviews(spark)
raw_df.show(5, truncate=False)


In [ ]:
# Stop all running streams
stop_all_streams(spark)
